In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/crimes_cleaned.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (2441506, 28)


,ID,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,...,DayOfWeek,DayOfWeekName,Hour,Quarter,WeekOfYear,IsWeekend,Lat_bin,Lon_bin,HourBin,Crime_Category
0,14135339,2026-03-13 00:00:00,075XX S KINGSTON AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,0,0,421,...,4,Friday,0,1,11,False,2087,-4379,0,Theft
1,14135179,2026-03-13 00:00:00,050XX N MARINE DR,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,0,0,2024,...,4,Friday,0,1,11,False,2098,-4383,0,Property
2,14138214,2026-03-13 00:00:00,075XX S STONY ISLAND AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,0,0,411,...,4,Friday,0,1,11,False,2087,-4380,0,Violent


In [2]:
if 'Lat_bin' not in df.columns:
    df['Lat_bin'] = (df['Latitude'] // 0.02).astype(int)
if 'Lon_bin' not in df.columns:
    df['Lon_bin'] = (df['Longitude'] // 0.02).astype(int)
if 'HourBin' not in df.columns:
    def get_hour_bin(h):
        if 0 <= h < 4: return 0
        if 4 <= h < 8: return 1
        if 8 <= h < 12: return 2
        if 12 <= h < 16: return 3
        if 16 <= h < 20: return 4
        return 5
    df['HourBin'] = df['Hour'].apply(get_hour_bin)
if 'Crime_Category' not in df.columns:
    CRIME_MAP = {'Violent': ['BATTERY','ASSAULT','ROBBERY','CRIMINAL SEXUAL ASSAULT','HOMICIDE','KIDNAPPING'],
                 'Theft': ['THEFT','MOTOR VEHICLE THEFT','BURGLARY'], 'Property': ['CRIMINAL DAMAGE','CRIMINAL TRESPASS','ARSON'],
                 'Narcotics': ['NARCOTICS','OTHER NARCOTIC VIOLATION'], 'Other': []}
    def map_cat(pt):
        for c, t in CRIME_MAP.items():
            if pt in t: return c
        return 'Other'
    df['Crime_Category'] = df['Primary Type'].apply(map_cat)

top_loc = df['Location Description'].value_counts().head(20).index.tolist()
df['Location_enc'] = df['Location Description'].apply(lambda x: x if x in top_loc else 'OTHER')
df['Location_enc'] = LabelEncoder().fit_transform(df['Location_enc'].astype(str))

# Sample for faster training
SAMPLE_SIZE = 200000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
print("Sample size:", len(df))

Sample size: 200000


In [3]:
df['Community_Area'] = df['Community Area'].fillna(-1).astype(int)

# Task 1: Crime Category
feat_crime = ['Beat', 'Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Domestic']
X1 = df[feat_crime].astype(float)
y1 = df['Crime_Category']

# Task 2: Top 25 Beats
top_beats = df['Beat'].value_counts().head(25).index.tolist()
mask_b = df['Beat'].isin(top_beats)
feat_beat = ['Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Community_Area']
df['Crime_Cat_enc'] = LabelEncoder().fit_transform(df['Crime_Category'])
feat_beat2 = ['Lat_bin', 'Lon_bin', 'Crime_Cat_enc', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Community_Area']
X2 = df.loc[mask_b, feat_beat2].astype(float)
y2 = df.loc[mask_b, 'Beat'].astype(str)

# Task 3: HourBin
feat_hour = ['Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc', 'IsWeekend']
X3 = df[feat_hour].astype(float)
X3['IsWeekend'] = X3['IsWeekend'].astype(int)
y3 = df['HourBin']

print("Task 1 (Crime Cat):", X1.shape, y1.value_counts().to_dict())
print("Task 2 (Beat):", X2.shape)
print("Task 3 (HourBin):", X3.shape)

Task 1 (Crime Cat): (200000, 8) {'Theft': 65691, 'Violent': 62847, 'Other': 37517, 'Property': 27019, 'Narcotics': 6926}
Task 2 (Beat): (32984, 8)
Task 3 (HourBin): (200000, 6)


In [8]:
EXPERIMENT_LOG = []

def log_experiment(task, model_name, params, accuracy, f1, notes=""):
    EXPERIMENT_LOG.append({
        'task': task, 'model': model_name, 'params': str(params),
        'accuracy': round(float(accuracy), 4), 'f1_weighted': round(float(f1), 4), 'notes': notes
    })

In [4]:
# === Step 1: Install and import ===
# Run: pip install lightgbm category_encoders imbalanced-learn
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [5]:
# === Step 2: Feature engineering function ===
def engineer_features(df):
    out = df.copy()
    # Interaction features (construct then label-encode)
    out['Beat_x_DayOfWeek'] = (out['Beat'].astype(str) + '_' + out['DayOfWeek'].astype(str))
    out['Beat_x_HourBin'] = (out['Beat'].astype(str) + '_' + out['HourBin'].astype(str))
    out['Crime_x_DayOfWeek'] = (out['Crime_Cat_enc'].astype(str) + '_' + out['DayOfWeek'].astype(str))
    out['LatLon_grid'] = (out['Lat_bin'].astype(str) + '_' + out['Lon_bin'].astype(str))
    out['Community_x_Month'] = (out['Community_Area'].astype(str) + '_' + out['Month'].astype(str))
    for col in ['Beat_x_DayOfWeek', 'Beat_x_HourBin', 'Crime_x_DayOfWeek', 'LatLon_grid', 'Community_x_Month']:
        out[col] = LabelEncoder().fit_transform(out[col].astype(str))

    # Target-encoded aggregate features (groupby transform)
    out['beat_crime_count'] = out.groupby('Beat')['Beat'].transform('count')
    out['beat_domestic_rate'] = out.groupby('Beat')['Domestic'].transform('mean')
    out['location_crime_count'] = out.groupby('Location_enc')['Location_enc'].transform('count')
    out['community_crime_count'] = out.groupby('Community_Area')['Community_Area'].transform('count')
    # beat_night_rate: 6-bin HourBin: 0=0-4, 5=20-24 are night
    out['beat_night_rate'] = out.groupby('Beat')['HourBin'].transform(
        lambda g: g.isin([0, 5]).mean()
    )
    return out

df = engineer_features(df)
print("engineer_features done. New columns added.")

engineer_features done. New columns added.


In [6]:
# === Step 3: Shared LightGBM runner ===
def run_lgbm_task(X, y_raw, task_name, lgb_params, n_folds=5, verbose=True):
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    n_classes = len(le.classes_)
    class_weights = len(y) / (n_classes * np.bincount(y, minlength=n_classes))
    sample_weights = class_weights[y]

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    accs, f1s = [], []
    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        sw_tr = sample_weights[tr_idx]

        mdl = lgb.LGBMClassifier(**lgb_params)
        mdl.fit(
            X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_te, y_te)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        pred = mdl.predict(X_te)
        accs.append(accuracy_score(y_te, pred))
        f1s.append(f1_score(y_te, pred, average='weighted', zero_division=0))

    mean_acc, std_acc = float(np.mean(accs)), float(np.std(accs))
    mean_f1, std_f1 = float(np.mean(f1s)), float(np.std(f1s))
    if verbose:
        print(f"{task_name}: Accuracy {mean_acc:.4f} ± {std_acc:.4f}, F1 (weighted) {mean_f1:.4f} ± {std_f1:.4f}")
    log_experiment(task_name, 'LightGBM_CV', lgb_params, mean_acc, mean_f1, 'StratifiedKFold CV')

    # Refit on full data (no eval_set, no early stopping)
    final_params = {**lgb_params}
    final_params.pop('callbacks', None)
    final_mdl = lgb.LGBMClassifier(**final_params)
    final_mdl.fit(X, y, sample_weight=sample_weights)
    return final_mdl, le

In [9]:
# === Step 4a–4c: Task 1 — Crime Category ===
# Merge rare categories (< 1% of data)
cat_counts = df['Crime_Category'].value_counts(normalize=True)
rare_cats = cat_counts[cat_counts < 0.01].index.tolist()
print(f"Merging {len(rare_cats)} rare categories into 'OTHER': {rare_cats}")
df['Crime_Category_merged'] = df['Crime_Category'].apply(
    lambda x: 'OTHER' if x in rare_cats else x
)
print("Merged distribution:")
print(df['Crime_Category_merged'].value_counts())

feat_crime_v2 = [
    'Beat', 'Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month',
    'Location_enc', 'Domestic', 'Community_Area', 'IsWeekend', 'Quarter',
    'Beat_x_DayOfWeek', 'Beat_x_HourBin', 'LatLon_grid', 'Community_x_Month',
    'beat_crime_count', 'beat_domestic_rate', 'location_crime_count',
    'community_crime_count', 'beat_night_rate',
]
X1 = df[feat_crime_v2].astype(float)
y1 = df['Crime_Category_merged']

crime_lgb_params = {
    'objective': 'multiclass',
    'num_class': y1.nunique(),
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    # FIX 2: removed class_weight='balanced' — with 5 merged classes and a
    # dominant majority it over-weights rare classes and hurts overall accuracy.
    # Add back only if minority class recall is a specific project requirement.
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}

model_crime, le_crime = run_lgbm_task(X1, y1, 'Crime_Category_v2', crime_lgb_params)

Merging 0 rare categories into 'OTHER': []
Merged distribution:
Crime_Category_merged
Theft        65691
Violent      62847
Other        37517
Property     27019
Narcotics     6926
Name: count, dtype: int64
Crime_Category_v2: Accuracy 0.4271 ± 0.0034, F1 (weighted) 0.4454 ± 0.0031


In [12]:
# === Step 4d: Feature importance (Task 1) ===
imp = pd.DataFrame({
    'feature': feat_crime_v2,
    'importance': model_crime.feature_importances_
}).sort_values('importance', ascending=False)
print(imp.to_string(index=False))
top5 = imp.head(5)['feature'].tolist()
highlight = ['Beat_x_HourBin', 'beat_crime_count', 'beat_night_rate']
in_top5 = [f for f in highlight if f in top5]
print(f"\nNew interaction features in top 5: {in_top5 if in_top5 else 'None'}")

              feature  importance
                Month       15311
    Community_x_Month       13216
         Location_enc       12606
            DayOfWeek       12386
              HourBin       11766
 location_crime_count       11120
          LatLon_grid        9357
     beat_crime_count        9318
   beat_domestic_rate        9127
      beat_night_rate        9117
     Beat_x_DayOfWeek        7762
                 Beat        6716
       Beat_x_HourBin        6011
              Lon_bin        5428
community_crime_count        5365
       Community_Area        3368
              Lat_bin        2479
             Domestic        2117
              Quarter        1622
            IsWeekend         808

New interaction features in top 5: None


In [13]:
def compute_baseline_and_ceiling(X, y, task_name):
    majority_acc = pd.Series(y).value_counts(normalize=True).iloc[0]
    if 'Beat' in X.columns and 'DayOfWeek' in X.columns:
        group_mode_acc = (
            pd.DataFrame({
                'y': np.array(y),
                'Beat': X['Beat'].values,
                'DayOfWeek': X['DayOfWeek'].values
            })
            .groupby(['Beat', 'DayOfWeek'])['y']
            .apply(lambda g: g.value_counts().iloc[0] / len(g))
            .mean()
        )
        print(f"\n{task_name}:")
        print(f"  Majority class baseline:          {majority_acc:.4f}")
        print(f"  Group-mode ceiling (Beat×Day):    {group_mode_acc:.4f}")
        print(f"  → Model < ceiling: more features can help")
        print(f"  → Model ≈ ceiling: inherent noise is the limit")

In [17]:
# === Step 5: Task 3 — HourBin (4-bin) ===
if 'HourBin4' not in df.columns:
    def hour_to_4bin(h):
        if 0 <= h < 6:   return 0  # night
        if 6 <= h < 12:  return 1  # morning
        if 12 <= h < 18: return 2  # afternoon
        return 3  # evening
    df['HourBin4'] = df['Hour'].apply(hour_to_4bin)

# FIX 3: removed Crime_x_DayOfWeek — it combines Crime_Cat_enc (which encodes
# hour-correlated patterns) with DayOfWeek, creating a circular dependency in CV.
# Beat_x_DayOfWeek is safe here because Beat is not the target for this task.
feat_hour_v2 = [
    'Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc',
    'IsWeekend', 'Lat_bin', 'Lon_bin', 'Community_Area',
    'Beat_x_DayOfWeek',
    'LatLon_grid',
    'beat_night_rate', 'location_crime_count', 'community_crime_count',
]
X3 = df[feat_hour_v2].astype(float)
y3 = df['HourBin4']
print("Random baseline (4 classes): 0.2500")
compute_baseline_and_ceiling(X3, y3, 'HourBin4_v2')

hour_lgb_params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    # FIX 3 cont.: removed class_weight='balanced' — HourBin4 bins are roughly
    # equal in size (~25% each) so balanced weighting adds noise with no benefit.
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}
model_hour, le_hour = run_lgbm_task(X3, y3, 'HourBin4_v2', hour_lgb_params)

Random baseline (4 classes): 0.2500

HourBin4_v2:
  Majority class baseline:          0.3156
  Group-mode ceiling (Beat×Day):    0.3498
  → Model < ceiling: more features can help
  → Model ≈ ceiling: inherent noise is the limit


KeyboardInterrupt: 

In [18]:
# === Step 6: Task 2 — Beat Prediction (top 50, single split) ===
top_beats = df['Beat'].value_counts().head(50).index.tolist()
mask_b = df['Beat'].isin(top_beats)

# FIX 1: removed Beat_x_DayOfWeek and Beat_x_HourBin from this feature set.
# Both are constructed as str(Beat) + '_' + str(...), which directly encodes
# the Beat label inside a predictor for the Beat task — producing fake 1.0 accuracy.
# beat_crime_count and beat_domestic_rate are safe: they encode beat behaviour
# (volume, character) without leaking the beat ID.
feat_beat_v2 = [
    'Lat_bin', 'Lon_bin', 'Crime_Cat_enc', 'HourBin', 'DayOfWeek', 'Month',
    'Location_enc', 'Community_Area', 'IsWeekend',
    'LatLon_grid', 'Community_x_Month',
    'beat_crime_count', 'beat_domestic_rate',
]
X2 = df.loc[mask_b, feat_beat_v2].astype(float)
y2 = df.loc[mask_b, 'Beat'].astype(str)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)
le_beat = LabelEncoder()
y2_tr_enc = le_beat.fit_transform(y2_tr)
y2_te_enc = le_beat.transform(y2_te)
n_cl = len(le_beat.classes_)

# FIX 1 cont.: no class_weight — top-50 beats are roughly balanced in count.
beat_lgb_params = {
    'objective': 'multiclass',
    'num_class': n_cl,
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 10,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}
model_beat = lgb.LGBMClassifier(**beat_lgb_params)
model_beat.fit(X2_tr, y2_tr_enc, callbacks=[lgb.log_evaluation(-1)])
pred_beat = model_beat.predict(X2_te)
acc_beat = accuracy_score(y2_te_enc, pred_beat)
f1_beat = f1_score(y2_te_enc, pred_beat, average='weighted', zero_division=0)
print(f"Beat_v2: Accuracy {acc_beat:.4f}, F1 (weighted) {f1_beat:.4f}")
# Sanity check: if acc > 0.95, re-examine feat_beat_v2 for any Beat-derived features
log_experiment('Beat_v2', 'LightGBM_single_split', beat_lgb_params, acc_beat, f1_beat, 'Top 50 beats, leakage removed')

Beat_v2: Accuracy 1.0000, F1 (weighted) 1.0000


In [19]:
# === Leakage sanity check — run after any new feature set ===
def check_for_leakage(X, y, task_name, threshold=0.5):
    le = LabelEncoder()
    # Always encode non-numeric y (e.g. Crime_Category strings) — dtype can be object/string/category
    y_enc = le.fit_transform(y) if not pd.api.types.is_numeric_dtype(y) else np.array(y)
    y_series = pd.Series(y_enc, name='target')
    X_reset = X.reset_index(drop=True)
    corrs = X_reset.corrwith(y_series).abs().sort_values(ascending=False)
    high = corrs[corrs > threshold]
    print(f"\n{task_name} — features correlated > {threshold} with target:")
    if len(high):
        print(high.to_string())
        print("  ^^^ Review these for potential leakage")
    else:
        print("  None found — looks clean")
    return corrs

check_for_leakage(X2, y2,   'Beat_v2')
check_for_leakage(X1, y1,   'Crime_Category_v2')
check_for_leakage(X3, y3,   'HourBin4_v2')


Beat_v2 — features correlated > 0.5 with target:
Lat_bin               0.738129
LatLon_grid           0.712598
Community_Area        0.707577
Community_x_Month     0.578156
beat_domestic_rate    0.562375
  ^^^ Review these for potential leakage

Crime_Category_v2 — features correlated > 0.5 with target:
  None found — looks clean

HourBin4_v2 — features correlated > 0.5 with target:
  None found — looks clean


Location_enc             0.057309
IsWeekend                0.047979
beat_night_rate          0.034801
DayOfWeek                0.032562
location_crime_count     0.016037
Beat_x_DayOfWeek         0.014932
Crime_Cat_enc            0.008904
community_crime_count    0.005287
Lat_bin                  0.004499
Community_Area           0.004125
Month                    0.003780
LatLon_grid              0.003637
Lon_bin                  0.001729
Beat                     0.000715
dtype: float64

In [20]:
# === Step 8: Final comparison table ===
exp_df = pd.DataFrame(EXPERIMENT_LOG)
print("\nAll experiments:")
print(exp_df.sort_values(['task', 'accuracy'], ascending=[True, False]).to_string(index=False))

print("\nBest model per task:")
best = exp_df.loc[exp_df.groupby('task')['accuracy'].idxmax()]
print(best[['task', 'model', 'accuracy', 'f1_weighted', 'notes']].to_string(index=False))


All experiments:
             task                 model                                                                                                                                                                                                                                                                                               params  accuracy  f1_weighted                         notes
          Beat_v2 LightGBM_single_split                {'objective': 'multiclass', 'num_class': 50, 'metric': 'multi_logloss', 'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'verbose': -1, 'random_state': 42, 'n_jobs': -1}    1.0000       1.0000 Top 50 beats, leakage removed
          Beat_v2 LightGBM_single_split                {'objective': 'multiclass', 'num_class': 50, 'metric': 'multi_logloss', 'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples':